In [ ]:
!pip install langchain-google-genai
!pip install langchain_core

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

from google.colab import userdata
gemini_api = userdata.get('gemini_api')

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", api_key = gemini_api, temperature = 0.4, max_retries = 3)

print("Gemini model initialized successfully!")

Gemini model initialized successfully!


**Without history**

In [ ]:
while True:
  user_input = input("You: ")
  if user_input.lower() == "exit":
    break
  result = llm.invoke(user_input)
  print(result.content)

You: exit


**With history but not formatted**

In [ ]:
history = []
while True:
  user_input = input("You: ")
  if user_input.lower() == "exit":
    break
  history.append(user_input)
  result = llm.invoke(history)
  history.append(result.content)
  print("AI: " + result.content)

You: hi
AI: Hi there! How can I help you today?
You: nothing how are you
AI: I don't have feelings or a physical state like humans do, so I can't be "good" or "bad" in the way you might be!

However, I'm fully operational and ready to assist you. Thanks for asking!

Is there anything I can help you with today?
You: nothing bye
AI: Alright, have a great day! Feel free to reach out if you need anything else later.
You: exit


history is there but not formatted for small numbers it is fine but when there is large conversation it will be confusing for model to differentiate what is what

In [ ]:
print(history)

['hi', 'Hi there! How can I help you today?', 'nothing how are you', 'I don\'t have feelings or a physical state like humans do, so I can\'t be "good" or "bad" in the way you might be!\n\nHowever, I\'m fully operational and ready to assist you. Thanks for asking!\n\nIs there anything I can help you with today?', 'nothing bye', 'Alright, have a great day! Feel free to reach out if you need anything else later.']


**History with formatted role using langchin**

1.   SystemMessage: It's like setting the context for the model, mostly it will be done only once at the beginning.
2.   HumanMessage: It's the user input.
3. AIMessage: it's the model output.

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

chat_history_formatted = []
chat_history_formatted.append(SystemMessage(content="You are a helpful assistant."))
while True:
  user_input = input("You: ")
  if user_input.lower() == "exit":
    break
  chat_history_formatted.append(HumanMessage(content=user_input))
  result = llm.invoke(chat_history_formatted)
  chat_history_formatted.append(AIMessage(content=result.content))
  print("AI: " + result.content)

You: hi
AI: Hi there! How can I help you today?
You: exit


In [ ]:
print(chat_history_formatted)

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='hi', additional_kwargs={}, response_metadata={}), AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


**Prompts without roles**

In [ ]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    template = "You are a helpful assistant that summarises about the {topic} in {number} of lines",
    input_variables = ["topic", "number"],
    validation = True
)


In [ ]:
topic = input("Enter a topic to summarize: ")
number_of_lines = int(input("Enter the number of lines for the summary: "))

prompt = template.format(topic = topic, number = number_of_lines)

print(prompt)

Enter a topic to summarize: NLP
Enter the number of lines for the summary: 20
You are a helpful assistant that summarises about the NLP in 20 of lines


In [ ]:
llm.invoke(prompt)

AIMessage(content='Natural Language Processing (NLP) is a field of Artificial Intelligence focused on enabling computers to understand, interpret, and generate human language. Its primary goal is to bridge the communication gap between humans and machines.\n\nHistorically, NLP relied on rule-based systems and statistical methods to analyze text and speech. Early tasks included part-of-speech tagging, parsing, and named entity recognition.\n\nThe advent of machine learning brought significant advancements, moving towards data-driven approaches. Deep learning, particularly neural networks, revolutionized the field, allowing models to learn complex patterns directly from vast amounts of text.\n\nRecurrent Neural Networks (RNNs) and Long Short-Term Memory (LSTMs) were crucial for sequential data like language. However, the Transformer architecture, introduced in 2017, marked a paradigm shift.\n\nTransformers enabled highly parallelizable training and captured long-range dependencies more e

**Prompts with roles**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate([
    ("system", "You are a helpful assistant that helps with summarization"),
    ("human", "You are a helpful assistant that summarises about the {topic} in {number} of lines")
])

In [ ]:
topic = input("Enter a topic to summarize: ")
number_of_lines = int(input("Enter the number of lines for the summary: "))

prompt = template.invoke({"topic": topic, "number": number_of_lines})

print(prompt)

Enter a topic to summarize: ml
Enter the number of lines for the summary: 8
messages=[SystemMessage(content='You are a helpful assistant that helps with summarization', additional_kwargs={}, response_metadata={}), HumanMessage(content='You are a helpful assistant that summarises about the ml in 8 of lines', additional_kwargs={}, response_metadata={})]


In [ ]:
result = llm.invoke(prompt)
print(result.content)

Machine Learning (ML) enables computers to learn from data without being explicitly programmed.
Instead of rigid rules, algorithms identify patterns and make predictions or decisions.
The process involves training a model on vast datasets, allowing it to "learn" relationships.
Key types include Supervised Learning (with labeled data for prediction), and Unsupervised Learning (finding hidden structures).
Reinforcement Learning is another paradigm, where an agent learns through trial and error in an environment.
ML excels at tasks like image recognition, natural language processing, and recommendation systems.
Its power lies in automating complex tasks, discovering insights, and adapting to new information.
Ultimately, ML aims to improve performance over time, driving innovation across countless industries.


**Message place holder**

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

template = ChatPromptTemplate([
    ("system", "You are a helpful assistant that helps in refund queries"),
    MessagesPlaceholder(variable_name = 'chat_history'),
    ("human", "{query}")
])

In [ ]:
chat_history = []

# --- Turn 1 ---
query1 = "When will refund credit?"
prompt1 = template.invoke({'query': query1, 'chat_history': chat_history})
result1 = llm.invoke(prompt1)
print("AI Response :", result1.content)

chat_history.append(HumanMessage(content=query1))
chat_history.append(AIMessage(content=result1.content))


AI Response 1: The timing for a refund to credit can vary, but here's a general breakdown and what to consider:

**General Timeframe:**

*   Most refunds typically take **5-10 business days** to process and appear on your bank or credit card statement.
*   Some can be faster (2-3 business days), while others, especially during peak seasons or with certain banks, might take up to **15 business days**.

**Factors That Influence the Timing:**

1.  **Your Bank or Credit Card Company:** Each financial institution has its own processing times for incoming refunds.
2.  **The Original Payment Method:**
    *   **Credit Card:** Often takes the longest due to the layers of processing between the merchant, their bank, and your credit card issuer.
    *   **Debit Card:** Can sometimes be faster than credit cards, but still depends on your bank.
    *   **PayPal/Other Digital Wallets:** Often quicker, sometimes within a few days, as the funds might stay within the platform.
    *   **Bank Transfer:

In [ ]:
# --- Turn 2 ---
query2 = "for amazon company"
prompt2 = template.invoke({'query': query2, 'chat_history': chat_history})
result2 = llm.invoke(prompt2)
print("AI Response :", result2.content)

# Log Turn 2 to history for future turns
chat_history.append(HumanMessage(content=query2))
chat_history.append(AIMessage(content=result2.content))

AI Response 2: Okay, for Amazon, the refund timing is generally quite reliable, but it still depends on the original payment method.

Amazon typically processes refunds very quickly once your return is received and inspected, or once a cancellation is confirmed. However, the time it takes for the money to actually appear in your account varies.

Here's a breakdown for Amazon refunds:

1.  **Amazon Gift Card Balance:**
    *   **Timeframe:** Usually **2-3 hours**, sometimes instantly, once Amazon processes the refund. This is the fastest method.

2.  **Credit/Debit Card:**
    *   **Timeframe:** **3-5 business days** after Amazon processes the refund.
    *   **Why it takes time:** While Amazon sends the refund quickly, your bank or credit card company needs to process it and post it to your statement. This can sometimes take up to 10 business days, especially with certain banks or during peak periods.

3.  **Bank Account (ACH/Direct Debit):**
    *   **Timeframe:** **5-7 business days*

In [ ]:
chat_history

[HumanMessage(content='When will refund credit?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="The timing for a refund to credit can vary, but here's a general breakdown and what to consider:\n\n**General Timeframe:**\n\n*   Most refunds typically take **5-10 business days** to process and appear on your bank or credit card statement.\n*   Some can be faster (2-3 business days), while others, especially during peak seasons or with certain banks, might take up to **15 business days**.\n\n**Factors That Influence the Timing:**\n\n1.  **Your Bank or Credit Card Company:** Each financial institution has its own processing times for incoming refunds.\n2.  **The Original Payment Method:**\n    *   **Credit Card:** Often takes the longest due to the layers of processing between the merchant, their bank, and your credit card issuer.\n    *   **Debit Card:** Can sometimes be faster than credit cards, but still depends on your bank.\n    *   **PayPal/Other Digital Wallets:** 